In [1]:
from retrieve import RAGRetriever
def test_syntax_search():
    # Initialize the retriever
    retriever = RAGRetriever()
    
    # Test query
    test_query = "immigrants are taking our jobs" 
    
    # Get results
    results = retriever.semantic_search(test_query,similarity_threshold=0.1,limit_per_layer=20)
    
    # Print results
    print(f"Found {len(results)} results for query: '{test_query}'")
    for i, result in enumerate(results):
        print(f"\nResult {i+1}:")
        print(f"ID: {result['id']}")
        print(f"Topic: {result['topic']}")
        print(f"Content: {result['content'][:150]}..." if len(result['content']) > 150 else result['content'])
        print(f"Quality Score: {result['quality_score']}")
        if result['counter_content']:
            print(f"Counter ID: {result['counter_id']}")
            print(f"Counter Content: {result['counter_content'][:150]}..." if len(result['counter_content']) > 150 else result['counter_content'])

if __name__ == "__main__":
    test_syntax_search()

Created vector index for HateContent embeddings
Found 20 results for query: 'immigrants are taking our jobs'

Result 1:
ID: 73
Topic: Immigration
Content:  Immigrants are a nationwide menace, bringing crimes and despair wherever they go. They drain our resources and crumble our culture with their entitle...
Quality Score: 7.8
Counter ID: 73
Counter Content:  Despite the dangerous and baseless stereotypes, immigrants contribute positively to our society. Contrary to your claims, immigrants are less likely ...

Result 2:
ID: 74
Topic: Immigration
Content:  Immigrants are not simply newcomers; they're an invasive force, flooding our border communities with un Convention-al crimes. Every day, we hear tale...
Quality Score: 8.3
Counter ID: 74
Counter Content:  Immigration does not lead to an "invasive force." Having open borders brings economic benefits through increased labor supply, tax revenue, and innov...

Result 3:
ID: 75
Topic: Immigration
Content:  Every day, our streets bear witnes

In [19]:
import pandas as pd 

df = pd.read_csv("/Users/erfanbayat/Downloads/Multitarget-CONAN.csv")
df.head()

,INDEX,HATE_SPEECH,COUNTER_NARRATIVE,TARGET,VERSION
0,0,Maybe the UN could talk to those asian and afr...,"You're right, ocean pollution is one of the ma...",other,V1
1,1,‘Juice (Jews) are worse that nukes.’ Caption t...,You'd rather be nuked then?,JEWS,V1
2,2,Foreigners on UK benefits leaps 41% in 5 years...,You seem to be worried about government expend...,MIGRANTS,V1
3,3,"Jews cannot be patriots, since their allegianc...",You can have parents and grandparents born els...,JEWS,V1
4,4,"Israel, Zionism and Jewish Supremacist control...",Wouldn't you say its colonialism from many dif...,JEWS,V1


In [28]:
df["TARGET"].unique()

array(['other', 'JEWS', 'MIGRANTS', 'WOMEN', 'POC', 'LGBT+', 'MUSLIMS',
       'DISABLED'], dtype=object)

In [29]:
df["TARGET"].value_counts()

TARGET
MUSLIMS     1335
MIGRANTS     957
WOMEN        662
LGBT+        617
JEWS         594
POC          352
other        266
DISABLED     220
Name: count, dtype: int64

In [30]:
from sklearn.utils import resample

# Create a subsample of 1000 rows with the same distribution of the 'TARGET' label
subsample = df.groupby("TARGET", group_keys=False).apply(
    lambda x: resample(x, replace=False, n_samples=int(1000 * len(x) / len(df)), random_state=42)
)

# Reset the index of the subsample
subsample = subsample.reset_index(drop=True)

# Display the subsample
subsample["TARGET"].value_counts()

/var/folders/75/98xy4z_n0dz808sky81718640000gn/T/ipykernel_24410/2798761790.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subsample = df.groupby("TARGET", group_keys=False).apply(


TARGET
MUSLIMS     266
MIGRANTS    191
WOMEN       132
LGBT+       123
JEWS        118
POC          70
other        53
DISABLED     43
Name: count, dtype: int64

In [31]:
subsample.to_csv("subsampled_data.csv", index=False)